In [82]:
import pandas as pd


In [74]:
df_old = pd.read_csv('../../../data/processed/realestate_clean_name_ppl.csv')
df = pd.read_csv('../../../data/processed/land_dataset_final_v3.csv')


In [78]:
df_combined = pd.concat([df, df_old], ignore_index=True)


In [ ]:
# df_combined.to_csv('../../../data/processed/combine_land_dataset_final_v1.csv', index=False)

In [66]:
df_old.head()

,price,land_area,address_line_2,latitude,longitude,price_per_m2,geometry,population,h_id,near_Koh_Pich_in_km,...,f_road,f_secondary,f_service,f_steps,f_tertiary,f_track,f_trunk,f_trunk_link,f_unclassified,f_unused
0,1100000.0,124.0,Chakto Mukh,11.57561,104.920250,8870.967742,POINT (104.92025 11.57561),16252.0,8865846aadfffff,3,...,0,0,1,0,1,0,0,0,0,0
1,680000.0,80.0,Boeng Keng Kang Ti Bei,11.55000,104.930000,8500.000000,POINT (104.93 11.55),7658.0,8865846ae9fffff,1,...,0,0,1,0,0,0,0,0,0,0
2,550000.0,66.0,Chey Chumneah,11.57561,104.920250,8333.333333,POINT (104.92025 11.57561),16252.0,8865846aadfffff,3,...,0,0,1,0,1,0,0,0,0,0
3,750000.0,116.0,Tonle Basak,11.54450,104.913586,6465.517241,POINT (104.913586 11.5445),23239.0,8865846ac7fffff,3,...,0,1,1,0,1,0,0,0,0,0
4,420000.0,65.0,Chrouy Changvar,11.58000,104.930000,6461.538462,POINT (104.93 11.58),5351.0,886584685bfffff,3,...,0,0,1,0,1,0,0,0,0,0


In [64]:
df_old.drop(columns=['index_right'], inplace=True)

In [50]:
df_old_cp = df_old[['price', 'land_area', 'address_line_2', 'latitude', 'longitude', 'price_per_m2']].copy()


In [51]:
df_old_cp

,price,land_area,address_line_2,latitude,longitude,price_per_m2
0,1100000.0,124.0,Chakto Mukh,11.575610,104.920250,8870.967742
1,680000.0,80.0,BKK 3,11.550000,104.930000,8500.000000
2,550000.0,66.0,Chey Chumneah,11.575610,104.920250,8333.333333
3,750000.0,116.0,Tonle Bassac,11.544500,104.913586,6465.517241
4,420000.0,65.0,Chroy Changvar,11.580000,104.930000,6461.538462
...,...,...,...,...,...,...
3369,18000.0,1400.0,Phnom Penh Thmey,11.575610,104.920250,12.857143
3372,3400000.0,400000.0,Preaek Aeng,11.522551,104.962474,8.500000
3376,35000.0,10000.0,Chroy Changvar,11.589674,104.925654,3.500000
3385,270000.0,270000.0,Kamboul,11.531222,104.776086,1.000000


In [75]:
# Step 1: Filter eligible h_id (more than 5 rows total)
h_id_counts = df['h_id'].value_counts()
eligible_h_ids = h_id_counts[h_id_counts > 5].index

# Step 2: Filter eligible rows
eligible = df[(df['price_per_m2'] < 4500) & (df['h_id'].isin(eligible_h_ids))]

# Step 3: Sample up to 10 per h_id
samples_per_group = eligible.groupby('h_id').apply(
    lambda x: x.sample(min(len(x), 10), random_state=42)
).reset_index(drop=True)

# Step 4: Try to sample 4000, and expand if needed
if len(samples_per_group) >= 5000:
    to_drop = samples_per_group.sample(n=5000, random_state=42)
else:
    to_drop = samples_per_group.copy()
    remaining = 5000 - len(to_drop)

    # Get remaining eligible rows not already in to_drop
    remaining_eligible = eligible.drop(to_drop.index)

    # Try to drop more from remaining (but still max 10 per h_id total)
    h_id_drop_counts = to_drop['h_id'].value_counts()

    # Only allow h_id that hasn't hit 10 yet
    can_drop_more = remaining_eligible[
        remaining_eligible['h_id'].map(lambda h: h_id_drop_counts.get(h, 0) < 10)
    ]

    # Count how many more each h_id can drop
    can_drop_more['drop_allowance'] = can_drop_more['h_id'].map(
        lambda h: 10 - h_id_drop_counts.get(h, 0)
    )

    # Group again and sample within allowance
    more_drops = (
        can_drop_more.groupby('h_id')
        .apply(lambda x: x.sample(n=min(len(x), 10 - h_id_drop_counts.get(x.name, 0))))
        .reset_index(drop=True)
    )

    # Sample only as many as needed
    to_add = more_drops.sample(n=min(remaining, len(more_drops)), random_state=42)
    to_drop = pd.concat([to_drop, to_add])

# Step 5: Drop those rows
df_dropped = df.drop(to_drop.index)


C:\Users\User\AppData\Local\Temp\ipykernel_17336\333708835.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  samples_per_group = eligible.groupby('h_id').apply(


In [76]:
df = df_dropped.copy()

In [77]:
df

,address_subdivision,address_locality,address_line_2,h_id,price_per_m2,land_area,price,longitude,latitude,near_Koh_Pich_in_km,...,f_road,f_secondary,f_service,f_steps,f_tertiary,f_track,f_trunk,f_trunk_link,f_unclassified,f_unused
4,Phnom Penh,Doun Penh,Chakto Mukh,8865846a39fffff,5442.34,200,1088468.00,104.958218,11.558388,1,...,0,0,0,0,0,0,0,0,0,0
16,Phnom Penh,Doun Penh,Srah Chak,8865846aa5fffff,4226.76,248,1048236.48,104.920556,11.584438,4,...,0,1,1,0,0,0,0,0,0,0
34,Phnom Penh,Chamkar Mon,Boeng Trabaek,8865846ac1fffff,3899.11,72,280735.92,104.920474,11.539412,2,...,0,0,1,0,1,0,0,0,0,0
64,Phnom Penh,Tuol Kouk,Tuek L'ak Ti Bei,8865846a95fffff,3813.96,76,289860.96,104.886224,11.558990,6,...,0,0,0,0,0,0,0,0,0,0
98,Phnom Penh,Doun Penh,Srah Chak,8865846aa7fffff,3580.15,234,837755.10,104.909105,11.589344,5,...,0,0,1,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9267,Phnom Penh,Chraoy Chongvar,Bak Kaeng,886586a699fffff,729.57,182,132781.74,104.929226,11.701893,16,...,0,0,0,0,0,0,0,0,0,0
9268,Phnom Penh,Chraoy Chongvar,Preaek Ta Sek,8865846995fffff,571.98,212,121259.76,104.899855,11.667508,13,...,0,0,0,0,0,0,0,0,0,0
9269,Phnom Penh,Praek Pnov,Ponsang,8865846d85fffff,260.40,134,34893.60,104.756877,11.633307,22,...,0,0,0,0,0,0,0,0,0,0
9270,Phnom Penh,Pur SenChey,Kantaok,8865846e31fffff,1093.83,230,251580.90,104.785133,11.523526,17,...,0,0,1,0,0,0,0,0,0,0


In [20]:
df['h_id'].value_counts()


h_id
8865846aebfffff    126
8865846ae7fffff    125
8865846ae3fffff    120
8865846ae1fffff    106
8865846ac7fffff     93
                  ... 
8865846f47fffff      1
8865846133fffff      1
88658478c9fffff      1
8865846a0dfffff      1
886586a68bfffff      1
Name: count, Length: 840, dtype: int64

In [13]:
h_id_counts = df['h_id'].value_counts().reset_index()
h_id_counts.columns = ['h_id', 'count']

In [83]:
df_combined = pd.read_csv('../../../data/processed/combine_land_dataset_final_v1.csv')

C:\Users\User\AppData\Local\Temp\ipykernel_17336\521291733.py:1: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_combined = pd.read_csv('../../../data/processed/combine_land_dataset_final_v1.csv')


In [101]:
df_combined.isnull().sum()

address_subdivision       0
address_locality       2399
address_line_2            0
price_per_m2              0
land_area                 0
                       ... 
f_unused                  0
geometry                  0
index_right              12
population               12
h_id                     12
Length: 236, dtype: int64

In [91]:
df_combined.head()

,address_subdivision,address_locality,address_line_2,price_per_m2,land_area,price,longitude,latitude,near_Koh_Pich_in_km,Koh_Pich_nearest,...,f_tertiary,f_track,f_trunk,f_trunk_link,f_unclassified,f_unused,geometry,index_right,population,h_id
0,Phnom Penh,Doun Penh,Chakto Mukh,5442.34,200.0,1088468.00,104.958218,11.558388,1,0,...,0,0,0,0,0,0,POINT (104.958218177304 11.55838750186468),53963.0,8.0,8865846a39fffff
3,Phnom Penh,Doun Penh,Srah Chak,4226.76,248.0,1048236.48,104.920556,11.584438,4,0,...,0,0,0,0,0,0,POINT (104.9205562206213 11.58443847258666),53924.0,29835.0,8865846aa5fffff
5,Phnom Penh,Chamkar Mon,Boeng Trabaek,3899.11,72.0,280735.92,104.920474,11.539412,2,0,...,1,0,0,0,0,0,POINT (104.9204743117011 11.53941192849337),53912.0,18375.0,8865846ac1fffff
9,Phnom Penh,Tuol Kouk,Tuek L'ak Ti Bei,3813.96,76.0,289860.96,104.886224,11.558990,6,0,...,0,0,0,0,0,0,POINT (104.8862240095462 11.55899002091164),53931.0,16116.0,8865846a95fffff
11,Phnom Penh,Doun Penh,Srah Chak,3580.15,234.0,837755.10,104.909105,11.589344,5,0,...,1,0,0,0,0,0,POINT (104.9091054452488 11.58934447384246),53923.0,36545.0,8865846aa7fffff


In [90]:
df_combined['address_subdivision'] = 'Phnom Penh'

In [99]:
df_combined.head(
)

,address_subdivision,address_locality,address_line_2,price_per_m2,land_area,price,longitude,latitude,near_Koh_Pich_in_km,Koh_Pich_nearest,...,f_tertiary,f_track,f_trunk,f_trunk_link,f_unclassified,f_unused,geometry,index_right,population,h_id
0,Phnom Penh,Doun Penh,Chakto Mukh,5442.34,200.0,1088468.00,104.958218,11.558388,1,0,...,0,0,0,0,0,0,POINT (104.958218177304 11.55838750186468),53963.0,8.0,8865846a39fffff
3,Phnom Penh,Doun Penh,Srah Chak,4226.76,248.0,1048236.48,104.920556,11.584438,4,0,...,0,0,0,0,0,0,POINT (104.9205562206213 11.58443847258666),53924.0,29835.0,8865846aa5fffff
5,Phnom Penh,Chamkar Mon,Boeng Trabaek,3899.11,72.0,280735.92,104.920474,11.539412,2,0,...,1,0,0,0,0,0,POINT (104.9204743117011 11.53941192849337),53912.0,18375.0,8865846ac1fffff
9,Phnom Penh,Tuol Kouk,Tuek L'ak Ti Bei,3813.96,76.0,289860.96,104.886224,11.558990,6,0,...,0,0,0,0,0,0,POINT (104.8862240095462 11.55899002091164),53931.0,16116.0,8865846a95fffff
11,Phnom Penh,Doun Penh,Srah Chak,3580.15,234.0,837755.10,104.909105,11.589344,5,0,...,1,0,0,0,0,0,POINT (104.9091054452488 11.58934447384246),53923.0,36545.0,8865846aa7fffff


In [120]:
address_hierarchy = {
  "Phnom Penh": {
    "Chamkar Mon": [
      "Boeng Keng Kang Ti Bei",
      "Boeng Keng Kang Ti Muoy",
      "Boeng Keng Kang Ti Pir",
      "Boeng Trabaek",
      "Olympic",
      "Phsar Daeum Thkov",
      "Tonle Basak",
      "Tumnob Tuek",
      "Tuol Svay Prey Ti Muoy",
      "Tuol Svay Prey Ti Pir",
      "Tuol Tumpung Ti Muoy",
      "Tuol Tumpung Ti Pir",
      "New Commune"
    ],
    "Chbar Ampov": [
      "Chbar Ampov Ti Pir",
      "Chhbar Ampov Ti Muoy",
      "Kbal Kaoh",
      "Nirouth",
      "Preaek Aeng",
      "Preaek Pra",
      "Preaek Thmei",
      "Veal Sbov"
    ],
    "Chraoy Chongvar": [
      "Bak Kaeng",
      "Chrouy Changvar",
      "Kaoh Dach",
      "Preaek Lieb",
      "Preaek Ta Sek"
    ],
    "Dangkao": [
      "Cheung Aek",
      "Dangkao",
      "Kong Noy",
      "Pong Tuek",
      "Preaek Kampues",
      "Prey Sa",
      "Prey Veaeng",
      "Roluos",
      "Sak Sampov",
      "Spean Thma",
      "Tien"
    ],
    "Doun Penh": [
      "Boeng Reang",
      "Chakto Mukh",
      "Chey Chummeah",
      "Phsar Chas",
      "Phsar Kandal Ti Muoy",
      "Phsar Kandal Ti Pir",
      "Phsar Thmei Ti Bei",
      "Phsar Thmei Ti Muoy",
      "Phsar Thmei Ti Pir",
      "Srah Chak",
      "Voat Phnum"
    ],
    "Mean Chey": [
      "Boeng Tumpun",
      "Chak Angrae Kraom",
      "Chak Angrae Leu",
      "Stueng Mean Chey"
    ],
    "Praek Pnov": [
      "Kouk Roka",
      "Ponhea Pon",
      "Ponsang",
      "Preaek Phnov"
    ],
    "Prampir Meakkakra": [
      "Boeng Proluet",
      "Mittapheap",
      "Monourom",
      "Ou Ruessei Ti Bei",
      "Ou Ruessei Ti Buon",
      "Ou Ruessei Ti Muoy",
      "Ou Ruessei Ti Pir",
      "Veal Vong"
    ],
    "Pur SenChey": [
      "Boeng Thum",
      "Chaom Chau",
      "Kakab",
      "Kamboul",
      "Kantaok",
      "Ovlaok",
      "Phleung Chheh Roteh",
      "Samraong Kraom",
      "Snaor",
      "Trapeang Krasang"
    ],
    "Russey Keo": [
      "Chrang Chamreh Ti Muoy",
      "Chrang Chamreh Ti Pir",
      "Kilomaetr Lekh Prammuoy",
      "Ruessei Kaev",
      "Svay Pak",
      "Tuol Sangke"
    ],
    "Saensokh": [
      "Khmuonh",
      "Krang Thnong",
      "Phnom Penh Thmei",
      "Tuek Thla"
    ],
    "Tuol Kouk": [
      "Boeng Kak Ti Muoy",
      "Boeng Kak Ti Pir",
      "Boeng Salang",
      "Phsar Daeum Kor",
      "Phsar Depou Ti Bei",
      "Phsar Depou Ti Muoy",
      "Phsar Depou Ti Pir",
      "Tuek L'ak Ti Bei",
      "Tuek L'ak Ti Muoy",
      "Tuek L'ak Ti Pir"
    ]
  }
}

In [121]:
flat_address_map = {
    subcommune.strip().lower(): district
    for district, sub_list in address_hierarchy["Phnom Penh"].items()
    for subcommune in sub_list
}


In [122]:
df_combined_1 = df_combined.copy()

In [123]:
df_combined_1["address_line_2_clean"] = df_combined_1["address_line_2"].str.strip().str.lower()


In [132]:
df_combined_1["address_locality"] = df_combined_1["address_locality"].fillna(
    df_combined_1["address_line_2_clean"].map(flat_address_map)
)

In [125]:
df_combined_1

,address_subdivision,address_locality,address_line_2,price_per_m2,land_area,price,longitude,latitude,near_Koh_Pich_in_km,Koh_Pich_nearest,...,f_track,f_trunk,f_trunk_link,f_unclassified,f_unused,geometry,index_right,population,h_id,address_line_2_clean
0,Phnom Penh,Doun Penh,Chakto Mukh,5442.340000,200.0,1088468.00,104.958218,11.558388,1,0,...,0,0,0,0,0,POINT (104.958218177304 11.55838750186468),53963.0,8.0,8865846a39fffff,chakto mukh
3,Phnom Penh,Doun Penh,Srah Chak,4226.760000,248.0,1048236.48,104.920556,11.584438,4,0,...,0,0,0,0,0,POINT (104.9205562206213 11.58443847258666),53924.0,29835.0,8865846aa5fffff,srah chak
5,Phnom Penh,Chamkar Mon,Boeng Trabaek,3899.110000,72.0,280735.92,104.920474,11.539412,2,0,...,0,0,0,0,0,POINT (104.9204743117011 11.53941192849337),53912.0,18375.0,8865846ac1fffff,boeng trabaek
9,Phnom Penh,Tuol Kouk,Tuek L'ak Ti Bei,3813.960000,76.0,289860.96,104.886224,11.558990,6,0,...,0,0,0,0,0,POINT (104.8862240095462 11.55899002091164),53931.0,16116.0,8865846a95fffff,tuek l'ak ti bei
11,Phnom Penh,Doun Penh,Srah Chak,3580.150000,234.0,837755.10,104.909105,11.589344,5,0,...,0,0,0,0,0,POINT (104.9091054452488 11.58934447384246),53923.0,36545.0,8865846aa7fffff,srah chak
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15065,Phnom Penh,Saensokh,Phnom Penh Thmei,12.857143,1400.0,18000.00,104.920250,11.575610,3,0,...,0,0,0,0,0,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,phnom penh thmei
15071,Phnom Penh,Chbar Ampov,Preaek Aeng,8.500000,400000.0,3400000.00,104.962474,11.522551,3,0,...,0,0,0,0,0,POINT (104.962474 11.522551),53953.0,3342.0,8865846a51fffff,preaek aeng
15073,Phnom Penh,Chraoy Chongvar,Chrouy Changvar,3.500000,10000.0,35000.00,104.925654,11.589674,4,0,...,0,1,1,0,0,POINT (104.925654 11.589674),54162.0,11327.0,8865846819fffff,chrouy changvar
15076,Phnom Penh,Pur SenChey,Kamboul,1.000000,270000.0,270000.00,104.776086,11.531222,18,0,...,0,0,0,0,0,POINT (104.776086 11.5312221),53668.0,1122.0,8865846e33fffff,kamboul


In [183]:
df_null = df_combined_1[df_combined_1['address_locality'].isnull()]


In [163]:
df_combined_1


,address_subdivision,address_locality,address_line_2,price_per_m2,land_area,price,longitude,latitude,near_Koh_Pich_in_km,Koh_Pich_nearest,...,f_track,f_trunk,f_trunk_link,f_unclassified,f_unused,geometry,index_right,population,h_id,address_line_2_clean
0,Phnom Penh,Doun Penh,Chakto Mukh,5442.340000,200.0,1088468.00,104.958218,11.558388,1,0,...,0,0,0,0,0,POINT (104.958218177304 11.55838750186468),53963.0,8.0,8865846a39fffff,chakto mukh
3,Phnom Penh,Doun Penh,Srah Chak,4226.760000,248.0,1048236.48,104.920556,11.584438,4,0,...,0,0,0,0,0,POINT (104.9205562206213 11.58443847258666),53924.0,29835.0,8865846aa5fffff,srah chak
5,Phnom Penh,Chamkar Mon,Boeng Trabaek,3899.110000,72.0,280735.92,104.920474,11.539412,2,0,...,0,0,0,0,0,POINT (104.9204743117011 11.53941192849337),53912.0,18375.0,8865846ac1fffff,boeng trabaek
9,Phnom Penh,Tuol Kouk,Tuek L'ak Ti Bei,3813.960000,76.0,289860.96,104.886224,11.558990,6,0,...,0,0,0,0,0,POINT (104.8862240095462 11.55899002091164),53931.0,16116.0,8865846a95fffff,tuek l'ak ti bei
11,Phnom Penh,Doun Penh,Srah Chak,3580.150000,234.0,837755.10,104.909105,11.589344,5,0,...,0,0,0,0,0,POINT (104.9091054452488 11.58934447384246),53923.0,36545.0,8865846aa7fffff,srah chak
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15065,Phnom Penh,Saensokh,Phnom Penh Thmei,12.857143,1400.0,18000.00,104.920250,11.575610,3,0,...,0,0,0,0,0,POINT (104.92025 11.57561),53920.0,16252.0,8865846aadfffff,phnom penh thmei
15071,Phnom Penh,Chbar Ampov,Preaek Aeng,8.500000,400000.0,3400000.00,104.962474,11.522551,3,0,...,0,0,0,0,0,POINT (104.962474 11.522551),53953.0,3342.0,8865846a51fffff,preaek aeng
15073,Phnom Penh,Chraoy Chongvar,Chrouy Changvar,3.500000,10000.0,35000.00,104.925654,11.589674,4,0,...,0,1,1,0,0,POINT (104.925654 11.589674),54162.0,11327.0,8865846819fffff,chrouy changvar
15076,Phnom Penh,Pur SenChey,Kamboul,1.000000,270000.0,270000.00,104.776086,11.531222,18,0,...,0,0,0,0,0,POINT (104.776086 11.5312221),53668.0,1122.0,8865846e33fffff,kamboul


In [185]:
df_combined_1.loc[
    df_combined_1['address_line_2'].isin(["Boeung Tumpun Pir"]),
    'address_locality'
] = 'Mean Chey'


In [191]:
df_combined_1.drop(columns='address_line_2_clean', inplace=True)

In [196]:
# Step 1: Group by 'h_id' and calculate the statistics
price_stats = df_combined_1.groupby('h_id')['price_per_m2'].agg(
    mean_price_per_m2='mean',
    max_price_per_m2='max',
    median_price_per_m2='median',
    min_price_per_m2='min'
).reset_index()

# Step 2: Merge the results back into the original DataFrame
df_combined_1 = df_combined_1.merge(price_stats, on='h_id', how='left')


In [202]:
df_combined_1.isnull().sum()

address_subdivision    0
address_locality       0
address_line_2         0
price_per_m2           0
land_area              0
                      ..
h_id                   0
mean_price_per_m2      0
max_price_per_m2       0
median_price_per_m2    0
min_price_per_m2       0
Length: 239, dtype: int64

In [203]:
df_combined_1.to_csv('../../../data/processed/combine_land_dataset_final_v1.csv', index=False)

In [184]:
df_null

,address_subdivision,address_locality,address_line_2,price_per_m2,land_area,price,longitude,latitude,near_Koh_Pich_in_km,Koh_Pich_nearest,...,f_track,f_trunk,f_trunk_link,f_unclassified,f_unused,geometry,index_right,population,h_id,address_line_2_clean
13359,Phnom Penh,NaN,Boeung Tumpun Pir,1354.166667,144.0,195000.0,104.920235,11.537783,2,0,...,0,0,0,0,0,POINT (104.920235 11.537783),53912.0,18375.0,8865846ac1fffff,boeung tumpun pir
14102,Phnom Penh,NaN,Boeung Tumpun Pir,888.888889,315.0,280000.0,104.908215,11.526125,4,0,...,0,0,0,0,0,POINT (104.9082148 11.52612525),53899.0,24762.0,8865846addfffff,boeung tumpun pir


In [174]:
unique_localities = df_combined_1['address_locality'].dropna().unique().tolist()


In [ ]:
unique_localities

['Doun Penh',
 'Chamkar Mon',
 'Tuol Kouk',
 'Saensokh',
 'Mean Chey',
 'Prampir Meakkakra',
 'Pur SenChey',
 'Chraoy Chongvar',
 'Dangkao',
 'Russey Keo',
 'Praek Pnov',
 'Chbar Ampov']